# Phase 2 — GEDI footprint download (Chapada dos Veadeiros)

Downloads GEDI L4A (biomass) and L2A (quality metrics) footprints, spatially
subsetted to the study area using NASA Harmony.

**Pipeline in this notebook**

| Step | What it does |
|------|--------------|
| 1 | Setup: config, Earth Engine, study area, native-vegetation mask |
| 2 | Export the study area as GeoJSON (Harmony needs it to clip) |
| 3 | Authenticate with NASA Earthdata |
| 4 | Connect to Harmony and resolve the product IDs |
| 5 | Define which variables and which year to request |
| 6 | Helper function: submit a request, wait, download |
| 7 | Submit + download L4A (biomass) — split into two requests |
| 8 | Submit + download L2A (quality metrics) |
| 9 | Verify what was downloaded |

**Why two products?** `agbd` (biomass, the target variable) is only in L4A.
`num_detectedmodes` (the most effective noise filter, Moudry et al. 2024) is only
in L2A. They are joined by `shot_number`, the unique footprint identifier.

**Why Harmony?** The full L4A archive over this area is ~56 GB for 2020-2022,
and almost all of it falls outside the study area. Harmony clips on NASA servers,
so only the relevant subset is downloaded.

Next notebook: `03_gedi_join_filter.ipynb` (read HDF5, join by shot_number, quality filter).

---
## 1. Setup

Run this cell first, always. If the kernel restarts, every variable is lost and
this cell rebuilds them.

In [1]:
from datetime import datetime
from glob import glob
from pathlib import Path
import json
import os

import ee
import h5py
import earthaccess
import requests
from harmony import Client, Collection, Request

from agb_cerrado.utils import load_config
from agb_cerrado.data import load_study_area, verify_area, load_mapbiomas_mask

# Project configuration (no hardcoded values below this point)
cfg = load_config("../config/config.yaml")

ee.Initialize(project=cfg["project"]["gee_project"])
print("Earth Engine connected")

# Study area: Chapada dos Veadeiros + buffer
sa = cfg["study_area"]
aoi = load_study_area(sa["clip_region_bbox"], sa["filter_name"], sa["buffer_m"])
area_km2 = verify_area(aoi, sa["area_km2_expected"])
print(f"Study area: {area_km2:.2f} km2")

# Native-vegetation mask (MapBiomas). Used later, in the filtering notebook.
native_mask = load_mapbiomas_mask(aoi, cfg["data"]["landcover_keep_classes"], year=2021)
print("Native-vegetation mask ready")

2026-07-22 19:26:39,570 | INFO | agb_cerrado.utils | Config loaded from ../config/config.yaml
2026-07-22 19:26:42,834 | INFO | agb_cerrado.data | AOI built (buffer=25000 m)


Earth Engine connected


2026-07-22 19:26:47,103 | INFO | agb_cerrado.data | AOI area verified: 11934.66 km2
2026-07-22 19:26:47,104 | INFO | agb_cerrado.data | Native-vegetation mask built (MapBiomas 2021)


Study area: 11934.66 km2
Native-vegetation mask ready


---
## 2. Export the study area as GeoJSON

Harmony clips the data using a polygon file. The geometry is simplified first:
the raw Earth Engine polygon contains near-duplicate vertices that Harmony
rejects (`Bad Request: The shape contained duplicate points`).

`maxError=100` means vertices may shift up to 100 m — negligible for a
~12,000 km2 area, and enough to remove the duplicates.

In [2]:
GEOJSON_PATH = "../data/interim/aoi_chapada.geojson"

Path(GEOJSON_PATH).parent.mkdir(parents=True, exist_ok=True)

aoi_simplified = aoi.simplify(maxError=100)
geometry = aoi_simplified.getInfo()

feature_collection = {
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {}, "geometry": geometry}],
}

with open(GEOJSON_PATH, "w") as f:
    json.dump(feature_collection, f)

n_vertices = len(geometry["coordinates"][0])
print(f"Saved: {GEOJSON_PATH}")
print(f"Vertices: {n_vertices}")

Saved: ../data/interim/aoi_chapada.geojson
Vertices: 332


---
## 3. NASA Earthdata authentication

Credentials are read from `~/.netrc`, never written in the notebook.

To create the file (run once, in a terminal, with your own credentials):

```bash
echo "machine urs.earthdata.nasa.gov login YOUR_USER password YOUR_PASSWORD" > ~/.netrc
chmod 600 ~/.netrc
```

In [3]:
auth = earthaccess.login(strategy="netrc")
print(f"Authenticated with NASA Earthdata: {auth.authenticated}")

2026-07-22 19:26:50,015 | INFO | earthaccess.auth | You're now authenticated with NASA Earthdata Login
2026-07-22 19:26:50,016 | INFO | earthaccess.auth | Using token with expiration date 09/12/2026


Authenticated with NASA Earthdata: True


---
## 4. Harmony client and product IDs

Each NASA dataset has a *concept ID*. It is resolved from the DOI so the
notebook does not depend on hardcoded identifiers that may change.

- L4A — Footprint Level Aboveground Biomass Density, `10.3334/ORNLDAAC/2056`
- L2A — Elevation and Height Metrics, `10.5067/GEDI/GEDI02_A.002`

In [4]:
harmony_client = Client()


def get_concept_id(doi: str) -> str:
    """Resolve a NASA CMR concept ID from a dataset DOI."""
    url = f"https://cmr.earthdata.nasa.gov/search/collections.json?doi={doi}"
    return requests.get(url).json()["feed"]["entry"][0]["id"]


concept_l4a = get_concept_id("10.3334/ORNLDAAC/2056")
concept_l2a = get_concept_id("10.5067/GEDI/GEDI02_A.002")

collection_l4a = Collection(id=concept_l4a)
collection_l2a = Collection(id=concept_l2a)

print(f"L4A: {concept_l4a}")
print(f"L2A: {concept_l2a}")

L4A: C2237824918-ORNL_CLOUD
L2A: C2142771958-LPCLOUD


---
## 5. Request definition

### Which year

Start with a single year to validate the whole pipeline end to end. Once the
join and the filter work, change `YEAR` and re-run to add 2021 and 2022.

### Which variables, and why

Every variable requested corresponds to a filtering criterion or to a required
piece of information. Nothing else is requested — the products contain 100+
variables, and requesting all of them would defeat the purpose of subsetting.

**L4A — biomass and its quality flags**

| Variable | Purpose |
|----------|---------|
| `agbd` | Target variable (aboveground biomass density) |
| `agbd_se` | GEDI's own standard error for each footprint |
| `agbd_pi_lower`, `agbd_pi_upper` | GEDI's own 90% prediction interval — baseline to compare against the CQR intervals |
| `l4_quality_flag` | Biomass quality filter (Kellner et al. 2023) |
| `l2_quality_flag` | Waveform quality filter |
| `algorithm_run_flag` | Confirms the L4A model actually ran for this shot |
| `predictor_limit_flag` | Model predictors outside the training range (extrapolation) |
| `response_limit_flag` | Model response outside the training range (extrapolation) |
| `degrade_flag` | Degraded positioning/pointing (Beck et al. 2021) |
| `sensitivity` | Beam penetration filter (Oliveira et al. 2023) |
| `solar_elevation` | Night-only filter (`< 0`) |
| `lat_lowestmode`, `lon_lowestmode` | Footprint coordinates |
| `shot_number` | Join key — included automatically by the subsetter |

The two extrapolation flags matter here specifically: Kellner et al. (2023)
report that dry tropical savannas are underrepresented in the L4A training
data. These flags mark where GEDI's own model is extrapolating, which is the
GEDI-side analogue of the Area of Applicability computed later for this study's
model.

**L2A — waveform quality metrics**

| Variable | Purpose |
|----------|---------|
| `shot_number` | Join key — must be requested explicitly for L2A |
| `num_detectedmodes` | Most effective noise filter (Moudry et al. 2024) |
| `elev_lowestmode` | GEDI ground elevation |
| `digital_elevation_model` | TanDEM-X reference, for the topographic filter |
| `quality_flag`, `sensitivity`, `solar_elevation`, `degrade_flag` | Cross-check against L4A |
| `lat_lowestmode`, `lon_lowestmode` | Footprint coordinates |

**Not requested — beam identity.** The power/coverage distinction is not a
variable: each beam is a separate HDF5 group, so the beam name is recovered
while reading the files. `BEAM0101`, `BEAM0110`, `BEAM1000` and `BEAM1011` are
the full power beams.

**Not requested — RH (canopy height) metrics.** L4A biomass is derived from
them, so using them as model predictors would be data leakage.

In [5]:
YEAR = 2020

temporal_range = {"start": datetime(YEAR, 1, 1), "stop": datetime(YEAR, 12, 31)}

BEAMS = [
    "BEAM0000", "BEAM0001", "BEAM0010", "BEAM0011",  # coverage beams
    "BEAM0101", "BEAM0110", "BEAM1000", "BEAM1011",  # full power beams
]


def per_beam(variables: list) -> list:
    """Expand variable names to the /BEAMXXXX/variable paths Harmony expects."""
    return [f"/{beam}/{var}" for beam in BEAMS for var in variables]





variables_l4a = per_beam([
    # Target variable and its uncertainty
    "agbd",
    "agbd_se",
    "agbd_pi_lower",           # GEDI's own 90% prediction interval
    "agbd_pi_upper",
    # Quality flags
    "l4_quality_flag",
    "l2_quality_flag",
    "algorithm_run_flag",      # L4A model actually ran for this shot
    "predictor_limit_flag",    # model extrapolating outside training range
    "response_limit_flag",
    "degrade_flag",
    "sensitivity",
    "solar_elevation",
    # Geolocation
    "lat_lowestmode",
    "lon_lowestmode",
])


variables_l2a = per_beam([
    "shot_number",
    "num_detectedmodes",
    "sensitivity",
    "solar_elevation",
    "degrade_flag",
    "elev_lowestmode",
    "digital_elevation_model",
    "quality_flag",
    "lat_lowestmode",
    "lon_lowestmode",
])

DOWNLOAD_DIR = f"../data/raw/gedi_{YEAR}"
Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

print(f"Year: {YEAR}")
print(f"L4A: {len(variables_l4a)} paths ({len(variables_l4a) // 8} variables x 8 beams)")
print(f"L2A: {len(variables_l2a)} paths ({len(variables_l2a) // 8} variables x 8 beams)")
print(f"Download directory: {DOWNLOAD_DIR}")

Year: 2020
L4A: 112 paths (14 variables x 8 beams)
L2A: 80 paths (10 variables x 8 beams)
Download directory: ../data/raw/gedi_2020


---
## 6. Submit and download

One function for all requests. Harmony processes the request on NASA servers
(this takes several minutes and shows a progress bar), then the clipped files
are downloaded.

`ignore_errors=True` lets Harmony skip individual problematic granules instead
of failing the whole job. A `Job is running with errors` message is expected
and not a failure.

The submit step is retried on HTTP 500 (`Internal Server Error`), which Harmony
returns intermittently under load. Malformed requests fail immediately instead,
so real errors are not hidden by the retry loop.

In [11]:
import time


def submit_and_download(collection, variables, product_label: str, max_retries: int = 5) -> list:
    """Submit a Harmony subset request and download the resulting files.

    Retries the submit step on transient server errors (HTTP 500), which Harmony
    returns intermittently under load. Malformed requests raise immediately.

    Args:
        collection: Harmony Collection object.
        variables: List of /BEAMXXXX/variable paths to request.
        product_label: Short name used for logging, e.g. "L4A".
        max_retries: How many times to retry the submit on server errors.

    Returns:
        List of paths to the downloaded *_subsetted.h5 files.
    """
    request = Request(
        collection=collection,
        variables=variables,
        temporal=temporal_range,
        shape=GEOJSON_PATH,
        ignore_errors=True,
    )

    job_id = None
    for attempt in range(1, max_retries + 1):
        try:
            job_id = harmony_client.submit(request)
            break
        except Exception as e:
            msg = str(e).lower()
            transient = "internal server error" in msg or "500" in msg
            if not transient or attempt == max_retries:
                raise
            wait = 30 * attempt
            print(f"[{product_label}] submit failed (attempt {attempt}/{max_retries}): {e}")
            print(f"[{product_label}] retrying in {wait}s...")
            time.sleep(wait)

    print(f"[{product_label}] job submitted: {job_id}")

    harmony_client.result_json(job_id, show_progress=True)
    print(f"[{product_label}] processing done, downloading...")

    futures = harmony_client.download_all(job_id, directory=DOWNLOAD_DIR, overwrite=True)
    files = [f.result() for f in futures]

    print(f"[{product_label}] {len(files)} files downloaded")
    return files

---
## 7. Download L4A (biomass)

Split into two requests. A single request with all 14 variables (112 variable
paths) returns HTTP 500 from Harmony regardless of retries; 56 paths per
request is accepted. Both parts cover the same orbit granules and both include
`shot_number`, so they are recombined in the join step.

In [13]:
# L4A split into two requests: 112 paths in one go returns HTTP 500
variables_l4a_a = per_beam([
    "agbd",
    "agbd_se",
    "l4_quality_flag",
    "l2_quality_flag",
    "degrade_flag",
    "lat_lowestmode",
    "lon_lowestmode",
])

variables_l4a_b = per_beam([
    "agbd_pi_lower",
    "agbd_pi_upper",
    "algorithm_run_flag",
    "predictor_limit_flag",
    "response_limit_flag",
    "sensitivity",
    "solar_elevation",
])

print(f"Part A: {len(variables_l4a_a)} paths")
print(f"Part B: {len(variables_l4a_b)} paths")

files_l4a_a = submit_and_download(collection_l4a, variables_l4a_a, "L4A-a")
files_l4a_b = submit_and_download(collection_l4a, variables_l4a_b, "L4A-b")

files_l4a = files_l4a_a + files_l4a_b
print(f"[L4A] total: {len(files_l4a)} files")

Part A: 56 paths
Part B: 56 paths
[L4A-a] job submitted: 260c59cd-b93e-4279-90bc-757cdf7e32d5


 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   9% ] |####                                               | [-]
 [ Processing:   9% ] |####                                               | [\]
 [ Processing:   9% ] |####                                               | [|]
 [ Processing:   9% ] |####             

[L4A-a] processing done, downloading...
../data/raw/gedi_2020/263786532_GEDI04_A_2020052181703_O06765_04_T04320_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786528_GEDI04_A_2020044212416_O06643_04_T05590_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786534_GEDI04_A_2020056164321_O06826_04_T04167_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786526_GEDI04_A_2020032134141_O06452_01_T03042_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786527_GEDI04_A_2020044090108_O06635_01_T03042_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786533_GEDI04_A_2020056042014_O06818_01_T01925_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786530_GEDI04_A_2020048195041_O06704_04_T01321_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786531_GEDI04_A_2020052055355_O06757_01_T00349_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786529_GEDI04_A_2020048072733_O06696_01_T00196_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263786535_GEDI04_A_2020060024632_O0687

 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                 

[L4A-b] processing done, downloading...
../data/raw/gedi_2020/263787312_GEDI04_A_2020072102816_O07070_04_T00204_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787316_GEDI04_A_2020080072040_O07192_04_T03050_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787315_GEDI04_A_2020079185724_O07184_01_T05077_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787302_GEDI04_A_2020052181703_O06765_04_T04320_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787305_GEDI04_A_2020060024632_O06879_01_T04618_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787306_GEDI04_A_2020060150939_O06887_04_T01474_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787301_GEDI04_A_2020052055355_O06757_01_T00349_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787297_GEDI04_A_2020044090108_O06635_01_T03042_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787317_GEDI04_A_2020083172447_O07245_01_T03348_02_002_02_V002_subsetted.h5
../data/raw/gedi_2020/263787298_GEDI04_A_2020044212416_O0664

../data/raw/gedi_2020/263801802_GEDI02_A_2020044090108_O06635_01_T03042_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801803_GEDI02_A_2020044212416_O06643_04_T05590_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801801_GEDI02_A_2020032134141_O06452_01_T03042_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801807_GEDI02_A_2020052181703_O06765_04_T04320_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801804_GEDI02_A_2020048072733_O06696_01_T00196_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801806_GEDI02_A_2020052055355_O06757_01_T00349_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801808_GEDI02_A_2020056042014_O06818_01_T01925_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801811_GEDI02_A_2020060150939_O06887_04_T01474_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801810_GEDI02_A_2020060024632_O06879_01_T04618_02_003_01_V002_subsetted.h5
../data/raw/gedi_2020/263801813_GEDI02_A_2020064133554_O06948_04_T04320_02_003_01_V002_subsetted.h5


---
## 8. Download L2A (quality metrics)

In [14]:
files_l2a = submit_and_download(collection_l2a, variables_l2a, "L2A")

[L2A] job submitted: 55b88f32-9999-4676-9c90-065c7e157bc1


 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                                                   | [/]
 [ Processing:   0% ] |                                                   | [-]
 [ Processing:   0% ] |                                                   | [\]
 [ Processing:   0% ] |                                                   | [|]
 [ Processing:   0% ] |                 

[L2A] processing done, downloading...
[L2A] 67 files downloaded


---
## 9. Verify the download

Files are grouped by the set of variables each one contains, which recovers the
Harmony jobs without relying on filenames. Three groups are expected: the two
L4A halves and L2A, each covering the same orbit granules.

The check that matters is `shot_number` in all three. Without it a group cannot
be joined to the others and would have to be requested again.

In [17]:
l4a_all = sorted(glob(os.path.join(DOWNLOAD_DIR, "*GEDI04_A*_subsetted.h5")))
l2a_all = sorted(glob(os.path.join(DOWNLOAD_DIR, "*GEDI02_A*_subsetted.h5")))

print(f"L4A files: {len(l4a_all)}")
print(f"L2A files: {len(l2a_all)}")


def first_populated_beam(h5_path: str):
    """Return (beam_name, variable_list) for the first beam containing data."""
    with h5py.File(h5_path, "r") as f:
        for beam in [k for k in f.keys() if k.startswith("BEAM")]:
            keys = list(f[beam].keys())
            if keys:
                return beam, keys
    return None, []


# Group files by their variable content: each Harmony job produced a distinct set
groups = {}
for path in l4a_all + l2a_all:
    _, variables = first_populated_beam(path)
    groups.setdefault(tuple(sorted(variables)), []).append(path)

print(f"\nDistinct variable sets: {len(groups)}\n")

for variables, files in groups.items():
    product = "L2A" if "num_detectedmodes" in variables else "L4A"
    print(f"{product} — {len(files)} files")
    print(f"    {', '.join(variables)}")
    print(f"  shot_number present: {'shot_number' in variables}\n")

L4A files: 134
L2A files: 67

Distinct variable sets: 3

L4A — 67 files
    agbd, agbd_se, degrade_flag, delta_time, l2_quality_flag, l4_quality_flag, lat_lowestmode, lon_lowestmode, shot_number
  shot_number present: True

L4A — 67 files
    agbd_pi_lower, agbd_pi_upper, algorithm_run_flag, delta_time, lat_lowestmode, lon_lowestmode, predictor_limit_flag, response_limit_flag, sensitivity, shot_number, solar_elevation
  shot_number present: True

L2A — 67 files
    degrade_flag, delta_time, digital_elevation_model, elev_lowestmode, lat_lowestmode, lon_lowestmode, num_detectedmodes, quality_flag, sensitivity, shot_number, solar_elevation
  shot_number present: True



---
## Next steps

1. **To add more years:** change `YEAR` in section 5, then re-run sections 5-9.
2. **To join and filter:** continue in `03_gedi_join_filter.ipynb`.

### References for the filtering criteria

- Beck, J., Armston, J., Hofton, M., & Luthcke, S. (2021). *GEDI Level 02 User Guide*. USGS EROS.
- Kellner, J. R., Armston, J., & Duncanson, L. (2023). Algorithm theoretical basis document for GEDI footprint aboveground biomass density. *Earth and Space Science*, 10(4).
- Moudry, V., et al. (2024). How to find accurate terrain and canopy height GEDI footprints in temperate forests and grasslands? *Earth and Space Science*, 11.
- Oliveira, P. V., Zhang, X., Peterson, B., & Ometto, J. P. (2023). Using simulated GEDI waveforms to evaluate the effects of beam sensitivity and terrain slope on GEDI L2A relative height metrics over the Brazilian Amazon Forest. *Science of Remote Sensing*, 7.